# 00 — Build LongVideoBench Cache (one-time)

**Purpose.** Download a subset of LongVideoBench, decode each video to up to 128 candidate frames at 1 fps, and save them as numpy arrays.

**Run this once.** When it finishes, save the notebook output as a Kaggle Dataset (`Save Version` → wait for completion → from the dataset's output, click `Save as Dataset`). Subsequent experiment notebooks mount that dataset read-only.

**Time budget.** Roughly 30–60 minutes for ~50 videos depending on download speed. Most of the time is the LFS download from HuggingFace.

**Prerequisites:**
1. Add your Hugging Face token as a Kaggle Secret named `HF_TOKEN`.
2. Accept the LongVideoBench dataset terms on HuggingFace.
3. Enable GPU (any tier — we don't really need GPU here, but Kaggle gives more RAM with GPU on).

## 1. Install dependencies and authenticate

In [ ]:
!pip install -q datasets huggingface_hub av pyarrow

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(hf_token)

## 2. Configuration

Start small. Once the pipeline works end-to-end you can rerun this notebook with a larger `N_QUESTIONS`. Each value below is intentionally exposed so you can experiment without editing code further down.

In [ ]:
# ===== CONFIG =====
N_QUESTIONS    = 30       # how many QA pairs to cache. Start with 30 for the smoke build.
FPS            = 1        # candidate frames per second
MAX_FRAMES     = 128      # cap per video (uniform sub-sample if longer)
FRAME_SIZE     = 384      # resize H = W = FRAME_SIZE (matches SigLIP / Qwen2-VL)
SEED           = 0
OUTPUT_ROOT    = "/kaggle/working/lvb-cache"
SPLIT          = "validation"
DATASET_REPO   = "longvideobench/LongVideoBench"

## 3. Pull the metadata (small, fast)

We grab the full validation metadata first so we know which video files we need. Only the videos referenced by our chosen QA subset get downloaded — saves significant time and disk.

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(DATASET_REPO, split=SPLIT)
df_full = ds.to_pandas()
print(f"Full {SPLIT} split: {len(df_full)} QA pairs")
df_full.head(2)

In [ ]:
# Sample a subset, stratified across duration so the cache spans all buckets.
import numpy as np

def duration_stratum(secs):
    if secs <= 15:   return 'a_short'
    if secs <= 60:   return 'b_med'
    if secs <= 600:  return 'c_long'
    return 'd_xlong'

df_full['stratum'] = df_full['duration'].apply(duration_stratum)
rng = np.random.default_rng(SEED)

per_stratum = max(1, N_QUESTIONS // 4)
picks = []
for s, group in df_full.groupby('stratum'):
    n = min(per_stratum, len(group))
    picks.append(group.sample(n=n, random_state=SEED))
df_sub = pd.concat(picks).reset_index(drop=True)
if len(df_sub) > N_QUESTIONS:
    df_sub = df_sub.sample(n=N_QUESTIONS, random_state=SEED).reset_index(drop=True)

print(f"Selected {len(df_sub)} QA pairs across {df_sub['video_id'].nunique()} unique videos")
print(df_sub.groupby('stratum').size())

## 4. Download only the needed videos

LongVideoBench distributes videos as tar parts. To avoid pulling all parts, we use `snapshot_download` with an `allow_patterns` filter built from the video IDs we actually need.

In [ ]:
import os
from huggingface_hub import snapshot_download

needed_videos = sorted(df_sub['video_id'].unique())
print(f"Need {len(needed_videos)} unique video files")

# LongVideoBench stores videos under `videos/<video_id>.mp4` after extraction.
# Sometimes the repo distributes them packed in tar parts. We try the direct
# per-file download first; if that fails we fall back to pulling tar parts.
patterns = [f"videos/{v}.mp4" for v in needed_videos]

video_dir = snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    allow_patterns=patterns,
    local_dir="/kaggle/working/lvb_videos_raw",
    local_dir_use_symlinks=False,
)
print("Snapshot dir:", video_dir)
found = [v for v in needed_videos if os.path.exists(os.path.join(video_dir, 'videos', f'{v}.mp4'))]
print(f"Got {len(found)}/{len(needed_videos)} videos via direct download")

In [ ]:
# Fallback: if direct files are missing, pull tar parts and extract.
# (Some HF dataset versions only ship the packed tarballs.)
missing = [v for v in needed_videos if not os.path.exists(os.path.join(video_dir, 'videos', f'{v}.mp4'))]

if missing:
    print(f"{len(missing)} missing — falling back to tar parts.")
    tar_dir = snapshot_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        allow_patterns=["videos.tar.part.*"],
        local_dir="/kaggle/working/lvb_tar",
        local_dir_use_symlinks=False,
    )
    import subprocess, glob
    parts = sorted(glob.glob(os.path.join(tar_dir, 'videos.tar.part.*')))
    print(f"Concatenating {len(parts)} tar parts...")
    subprocess.run(f"cat {' '.join(parts)} > /kaggle/working/videos.tar", shell=True, check=True)
    print("Extracting (this can take a few minutes)...")
    subprocess.run(
        "tar -xf /kaggle/working/videos.tar -C /kaggle/working/lvb_videos_raw",
        shell=True, check=True,
    )
    # Free disk
    subprocess.run("rm -f /kaggle/working/videos.tar", shell=True)
    found = [v for v in needed_videos if os.path.exists(os.path.join('/kaggle/working/lvb_videos_raw', 'videos', f'{v}.mp4'))]
    print(f"After fallback: {len(found)}/{len(needed_videos)} videos available")

video_root = "/kaggle/working/lvb_videos_raw/videos"

## 5. Decode each video to candidate frames

For each video, sample frames at 1 fps. If that produces more than `MAX_FRAMES`, uniformly sub-sample to `MAX_FRAMES`. Resize to `FRAME_SIZE × FRAME_SIZE` and save as `.npy` (uint8).

Why 384×384: SigLIP base uses 384-resolution patches, and Qwen2-VL handles this resolution efficiently. Bigger isn't useful for the selector; smaller hurts SigLIP quality.

In [ ]:
import av
import cv2
from pathlib import Path
from tqdm.auto import tqdm

frames_out = Path(OUTPUT_ROOT) / "frames"
frames_out.mkdir(parents=True, exist_ok=True)

def decode_video(path: str, fps: int, max_frames: int, size: int) -> np.ndarray:
    """Return uint8 array (N, size, size, 3) of candidate frames."""
    container = av.open(path)
    stream = container.streams.video[0]
    src_fps = float(stream.average_rate) if stream.average_rate else 30.0
    step = max(1, round(src_fps / fps))

    raw_frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i % step == 0:
            img = frame.to_ndarray(format="rgb24")
            raw_frames.append(img)
    container.close()

    if len(raw_frames) > max_frames:
        idx = np.linspace(0, len(raw_frames) - 1, max_frames).round().astype(int)
        raw_frames = [raw_frames[i] for i in idx]

    out = np.stack([
        cv2.resize(f, (size, size), interpolation=cv2.INTER_AREA)
        for f in raw_frames
    ]).astype(np.uint8)
    return out


decoded_ids = set()
failed_ids = []
for vid in tqdm(needed_videos, desc="Decoding"):
    out_path = frames_out / f"{vid}.npy"
    if out_path.exists():
        decoded_ids.add(vid)
        continue
    src = os.path.join(video_root, f"{vid}.mp4")
    if not os.path.exists(src):
        failed_ids.append(vid)
        continue
    try:
        arr = decode_video(src, FPS, MAX_FRAMES, FRAME_SIZE)
        np.save(out_path, arr)
        decoded_ids.add(vid)
    except Exception as e:
        print(f"[FAIL] {vid}: {e}")
        failed_ids.append(vid)

print(f"Decoded: {len(decoded_ids)}  Failed: {len(failed_ids)}")

## 6. Save metadata and clean up

We keep only the columns the runner needs. `options` becomes five separate columns (`option0..option4`) for parquet compatibility.

In [ ]:
df_sub = df_sub[df_sub['video_id'].isin(decoded_ids)].reset_index(drop=True)

# Normalize option columns to option0..option4 (pad short option lists).
keep_cols = ['video_id', 'question', 'correct_choice', 'duration', 'question_category']
for i in range(5):
    col = f'option{i}'
    if col not in df_sub.columns:
        df_sub[col] = None
    keep_cols.append(col)
df_out = df_sub[keep_cols]

meta_path = Path(OUTPUT_ROOT) / "metadata.parquet"
df_out.to_parquet(meta_path, index=False)
print(f"Wrote {len(df_out)} rows to {meta_path}")
df_out.head(3)

In [ ]:
# Clean up raw video files to keep the Kaggle Dataset slim.
# (The cache only needs metadata.parquet and frames/*.npy)
import shutil
for p in ["/kaggle/working/lvb_videos_raw", "/kaggle/working/lvb_tar"]:
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f"Removed {p}")

# Show final cache size
!du -sh /kaggle/working/lvb-cache
!ls /kaggle/working/lvb-cache
!ls /kaggle/working/lvb-cache/frames | head

## 7. Next step

1. Click **Save Version** in the top-right (commit the notebook).
2. When it finishes, open this notebook's output, click `New Dataset` and name it `lvb-cache` (or whatever you prefer).
3. In notebook `01_run_experiment.ipynb`, attach that dataset as input. The runner mounts it from `/kaggle/input/<your-dataset-name>/lvb-cache`.